## Binary-to-Decimal Algorithms for Cellular Automata
Some examples, concepts and algorithms for the module on Cellular Automata.

In [1]:
import matplotlib.pyplot as plt
from timeit import timeit


In the `week-05-CA` workbook we introduce an algorithm to apply a CA 1D state transition "rule".
1. A CA "rule" is just a lookup table - we lookup the value in the next timestep based on a cell's 3-bit "neighbourhood".
We know there are 256 such rules.  But which rule is this one?

In [2]:
def ca_1D_rule(neighbours: list) -> int:
    """ A "state transition rule" for computing the next CA state for one cell in the 3-cell neighbourhood. """
    return sum(neighbours) % 2

all_possible_neighbourhoods = (
    (0,0,0),
    (0,0,1),
    (0,1,0),
    (0,1,1),
    (1,0,0),
    (1,0,1),
    (1,1,0),
    (1,1,1),
)
# the same "rule", represented as an 8 bit vector (list of 0,1)
mod_2_transitions = [ca_1D_rule(n) for n in all_possible_neighbourhoods]
print('Transition map: shows how the rule transforms each input "neighbourhood" to the next state value of centre cell')
{neighbourhood: output for neighbourhood, output in zip(all_possible_neighbourhoods, mod_2_transitions)}

Transition map: shows how the rule transforms each input "neighbourhood" to the next state value of centre cell


{(0, 0, 0): 0,
 (0, 0, 1): 1,
 (0, 1, 0): 1,
 (0, 1, 1): 0,
 (1, 0, 0): 1,
 (1, 0, 1): 0,
 (1, 1, 0): 0,
 (1, 1, 1): 1}

2. But what is the Wolfram rule number?

Wolfram numbered the CA rules by treating this 8-bit transition vector as a binary number, where each bit represents one power-of-2.

In [3]:
rule_number = sum(2 ** i * mod_2_transitions[i] for i in range(8))
f'The "sum mod 2" rule, with transitions {mod_2_transitions}, is rule number: {rule_number}'

'The "sum mod 2" rule, with transitions [0, 1, 1, 0, 1, 0, 0, 1], is rule number: 150'

Notice if we reverse the "bits" of the transition vector, "10010110", the resulting "bit string" is the binary number equivalent to the rule name!

The "reverse" here is because lists (and vectors!) are indexed from "left-to-right", but the "places" in our number systems, like decimal and binary, increase right-to-left.  Note that these are just conventions - there's nothing meaningful about these ordering, they just happen to use opposite conventions.

In [4]:
rule_number = ''.join(str(bit) for bit in reversed(mod_2_transitions))
f"{rule_number} (base-2) == 150" "  (i.e., 2^7 + 2^4 + 2^2 + 2^1)"

'10010110 (base-2) == 150  (i.e., 2^7 + 2^4 + 2^2 + 2^1)'

3. Wait!  That algorithm is just a general-purpose binary-to-decimal conversion.  So going one step further....


In [5]:
def binary_to_decimal(bits: list[int]) -> int:
    """Convert bit string (a list of 0,1) to decimal equivalent"""
    places = reversed(range(len(bits)))  # process digit "places" in reverse so 2^0 is right-most bit
    powers_of_2 = (2**p for p in places)
    return sum(p * b for b, p in zip(bits, powers_of_2))

assert binary_to_decimal([1,0,1,0]) == 10  # 2^3 + 0 + 2^1 + 0
assert binary_to_decimal(list(reversed(mod_2_transitions))) == 150
"passed"

'passed'

4. But how to get the bit-string rule from its number?

We just need the opposite transformation - convert decimal to binary!

In [6]:
def decimal_to_binary(d: int) -> list[int]:
    """Generate base-2 bit-string for the base-10 integer d"""
    bits = []
    while d > 0:
        bits.insert(0, d % 2)  # insert next power-of-2 "bit" at front of list
        d //= 2                # remove same quantity from d
    return bits

assert decimal_to_binary(10) == [1,0,1,0]
assert decimal_to_binary(150) == [1, 0, 0, 1, 0, 1, 1, 0]
"passed"

'passed'

5. Finally!  To get the transition vector, we just need to reverse that bit-string so the lookup for neighbourhood (0,0,0) is at index 0 (start of list, on the left)

In [7]:
def get_ca_transition_vector(rule:int) -> list[int]:
    """
    Return the 8-bit transition vector for the given elementary CA rule number.
    rule: int 0-255
    returns: transition lookup vector - an 8-bit vector
    """
    bits = list(reversed(decimal_to_binary(rule)))
    return [0] * (8-len(bits)) + bits  # pad with zeros to make 8 bits

assert get_ca_transition_vector(0) == [0,0,0,0,0,0,0,0]
assert get_ca_transition_vector(255) == [1,1,1,1,1,1,1,1]
assert get_ca_transition_vector(150) == [0,1,1,0,1,0,0,1]
"passed"

'passed'

## Computing CA state transitions using the transition vector
The ultimate goal of all this was to create a data structure that makes it efficient to compute the state transitions for a 1D Cellular Automata.  Let's look at a couple ways we could achieve this...

1. A CA state transition is just a mapping from a 3-bit neighbourhood to the next state value for its centre cell.  So a natural data structure would be the dictionary we saw way above...

In [8]:
rule = 150

transition_map = {neighbourhood: output for neighbourhood, output in zip(all_possible_neighbourhoods, get_ca_transition_vector(rule))}
transition_map

{(0, 0, 0): 0,
 (0, 0, 1): 1,
 (0, 1, 0): 1,
 (0, 1, 1): 0,
 (1, 0, 0): 1,
 (1, 0, 1): 0,
 (1, 1, 0): 0,
 (1, 1, 1): 1}

2. applying the rule is super simple - just lookup the neighbourhood in the dict - an O(1) operation!

In [9]:
def get_next_state_from_dict(ca_state: list, lookup: dict) -> list:
    """ Return the next state for the CA using the given transition map."""
    # generate the 3-cell neighbourhoods
    neighbourhoods = ((ca_state[i-1], ca_state[i], ca_state[i+1]) for i in range(1, len(ca_state)-1))
    return [lookup[neighbours] for neighbours in neighbourhoods]

start_state = [1,0,0,1,0,1,1,0,1,0,0,0,1]  # arbitrary CA start state
get_next_state_from_dict(start_state, transition_map)

[1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1]

This seems like a very clean, simple approach to me.  It uses the efficiency of python dictionaries and is a very direct representation of the problem.  The textbook introduces one additional "refinement", so let's keep going to see if we can make sense of it...

3. Notice that we can interpret each 3-cell neighbourhood as a binary number between 0 (0,0,0) and 7 (1,1,1)  These map directly onto the indexes of the transition vector.  The approach the textbook uses is to use this fact to lookup the transition by indexing the vector directly - no dictionary required...

In [13]:
def get_next_state_from_vector(ca_state: list, lookup: list) -> list:
    """ Return the next state for the CA using the given transition vector."""
    # generate the 3-cell neighbourhoods
    transition_index = (binary_to_decimal((ca_state[i-1], ca_state[i], ca_state[i+1])) for i in range(1, len(ca_state)-1))
    return [lookup[index] for index in transition_index]


start_state = [1,0,0,1,0,1,1,0,1,0,0,0,1]  # arbitrary CA start state
get_next_state_from_vector(start_state, get_ca_transition_vector(rule))

[1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1]

#### Check our work!
Verify that the output from the `dict` lookup matches the output from the transition vector lookup.

##### What do you think?
The textbook treats each 3-cell neighbourhood as a 3-bit binary number, then uses that number to index the transition vector to lookup the next state. Compare this with the version where we just use each 3-cell state itself as the keys in a dict, and do the lookup directly.
What are the advanatages or disadvantages or each approach?
Could you write a small timing test to compare the relative performance of these 2 methods?

## Optional (advanced topic)

6. Efficient Implementation using Bit Shifting

The approach above gives us the concept. But, by recognizing that an `int` is already stored in binary inside the computer, we can implement `get_ca_transition_vector` more efficiently using bitwise operators.

The **bit shift** (`>>`) and **bitwise AND** (`&`) can be used to extract individual bits from an integer.
* `rule >> i`: shifts the $i$-th bit of `rule` to the "ones" place (the right-most bit).  This operation is equivalent to integer divide by $2^i$, but much more efficient.
* `& 1`: masks all but the right-most bit, leaving only the bit at that position - a single 0 or 1.  This operation is equivalent to `% 2`, but more efficient.

This allows us to construct the transition vector in a single pass without any intermediate transformations and expensive multiplications and divisions.


In [10]:
def get_ca_transition_vector_efficient(rule: int) -> list[int]:
    """
    Return the 8-bit transition vector for the given elementary CA rule number.
    Uses bit shifting for a more efficient implementation.
    rule: int 0-255
    returns: transition lookup vector - an 8-bit vector
    """
    return [(rule >> i) & 1 for i in range(8)]

assert get_ca_transition_vector_efficient(0) == [0,0,0,0,0,0,0,0]
assert get_ca_transition_vector_efficient(255) == [1,1,1,1,1,1,1,1]
assert get_ca_transition_vector_efficient(150) == [0,1,1,0,1,0,0,1]
"passed"

'passed'

Let's run some timing tests to see if that actually improves performance...

In [11]:
n_trials = 100

timings = {
    "basic": timeit(lambda: [get_ca_transition_vector(n) for n in range(256)],  number=n_trials),
    "efficient": timeit(lambda: [get_ca_transition_vector_efficient(n) for n in range(256)],  number=n_trials)
}
timings

{'basic': 0.021057415999999662, 'efficient': 0.017421041000000415}

##### What do you think?
- Under what circumistances might this efficiency gain be significant / worth wile?
- What are the tradeoffs between this compact 1-line algorithm vs. the original algorithm?
- Which algorithm do you prefer?